# ProteinMPNN — 8-mer Design with ETGE Locked

Generates 150 peptide candidates (8 residues) with ETGE motif locked.
ProteinMPNN redesigns the 4 free positions to optimize KEAP1 binding.

**Template:** DEETGEFL (positions 9-16 of Nrf2 in 2FLU)
**Locked:** ETGE at positions 3-6
**Free:** positions 1, 2, 7, 8

In [ ]:
# ============================================================
# CELL 1: Install + Clone
# ============================================================
%cd /content
!rm -rf NewPipeline_07-14
!git clone https://github.com/PMONESKIN/NewPipeline_07-14.git
%cd NewPipeline_07-14
!pip install -q pyyaml biopython torch
print("Ready.")

In [ ]:
# ============================================================
# CELL 2: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ============================================================
# CELL 3: Setup ProteinMPNN
# ============================================================
!python -u modules/02_design/setup_proteinmpnn.py

In [ ]:
# ============================================================
# CELL 4: Configure + Run Full Design
# ============================================================
import yaml
import json
import sys
import os
import importlib.util
import shutil

# Write config for 8-mers with ETGE locked
config = {
    'project': {'name': 'PeptideScreen', 'version': '2.0'},
    'targets': [{
        'name': 'Nrf2/KEAP1',
        'pdb_id': '2FLU',
        'receptor_chain': 'X',
        'ligand_chain': 'P',
        'fixed_motif': 'ETGE',
        'design_chain_length': 8,
        'mpnn_temperatures': [0.1, 0.2, 0.3],
    }],
    'candidates': {
        'initial_pool': 50,
        'final_shortlist': 10,
        'length': {'min': 5, 'max': 30},
        'filters': {'max_molecular_weight': 2500, 'flag_aggregation_prone': True},
    },
    'outputs': {
        'structures': 'data/structures/',
        'candidates': 'data/candidates/',
        'docking': 'data/docking/',
        'processed': 'data/processed/',
        'reports': 'outputs/reports/',
        'figures': 'outputs/figures/',
    },
}
with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("Config saved: 8-mer with ETGE locked")

# Helper to load modules from numbered directories
def load_mod(path):
    p = f"/content/NewPipeline_07-14/{path}"
    spec = importlib.util.spec_from_file_location("mod", p)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# Create run
sys.path.insert(0, '/content/NewPipeline_07-14')
from modules.run_manager import RunManager
rm = RunManager()
RUN_DIR = str(rm.run_dir)

# Save RUN_DIR for later cells
with open('/content/current_run.txt', 'w') as f:
    f.write(RUN_DIR)

print(f"\nRun: {RUN_DIR}")

# Phase 0: Fetch PDB + Analyze Interface
print("\n--- Fetching PDB ---")
load_mod("modules/01_targets/fetch_structures.py").run(run_dir=RUN_DIR)

print("\n--- Analyzing Interface ---")
load_mod("modules/01_targets/analyze_interface.py").run(run_dir=RUN_DIR)

# Phase 1: Extract 8-mer backbone centered on ETGE
print("\n--- Extracting 8-mer backbone centered on ETGE ---")
load_mod("modules/02_design/backbone_extract.py").run(run_dir=RUN_DIR, length=8)

# Phase 2: ProteinMPNN design
print("\n--- Running ProteinMPNN (50 seqs x 3 temps = 150 total) ---")
load_mod("modules/02_design/design_peptides.py").run(run_dir=RUN_DIR)

# Phase 3: Build candidate pool
print("\n--- Building candidate pool ---")
load_mod("modules/02_design/candidate_pool.py").run(run_dir=RUN_DIR)

print("\nDesign complete.")

In [ ]:
# ============================================================
# CELL 5: View All Candidates
# ============================================================
import json

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

candidates = json.load(open(f"{RUN_DIR}/candidates/candidate_pool.json"))

print(f"{'='*60}")
print(f"  {len(candidates)} unique 8-mer candidates with ETGE locked")
print(f"{'='*60}")
print(f"")
print(f"  {'ID':<16} {'Sequence':<12} {'MPNN Score':<12} {'Has ETGE'}")
print(f"  {'-'*16} {'-'*12} {'-'*12} {'-'*8}")

etge_count = 0
for c in candidates:
    has_etge = 'ETGE' in c['sequence']
    if has_etge:
        etge_count += 1
    mpnn = f"{c['mpnn_score']:.4f}" if c.get('mpnn_score') else 'seed'
    print(f"  {c['id']:<16} {c['sequence']:<12} {mpnn:<12} {'YES' if has_etge else 'NO'}")

print(f"\n  ETGE preserved: {etge_count}/{len(candidates)}")
print(f"  Template: DEETGEFL (from Nrf2 positions 9-16)")
print(f"  Locked: ETGE at positions 3-6")
print(f"  Free: positions 1, 2, 7, 8")

In [ ]:
# ============================================================
# CELL 6: Save to Google Drive
# ============================================================
import shutil
import os

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

dest = '/content/drive/MyDrive/PeptideScreen_Runs/8mer_ETGE_candidates'
os.makedirs(dest, exist_ok=True)

# Copy candidate pool
shutil.copy(f"{RUN_DIR}/candidates/candidate_pool.json", f"{dest}/candidate_pool.json")

# Copy interface data
for f_name in os.listdir(f"{RUN_DIR}/processed"):
    shutil.copy(f"{RUN_DIR}/processed/{f_name}", f"{dest}/{f_name}")

# Save summary
import json
candidates = json.load(open(f"{RUN_DIR}/candidates/candidate_pool.json"))
summary = {
    'total_candidates': len(candidates),
    'design_length': 8,
    'fixed_motif': 'ETGE',
    'template': 'DEETGEFL',
    'sequences': [{'id': c['id'], 'sequence': c['sequence'], 'mpnn_score': c.get('mpnn_score')} for c in candidates],
}
with open(f"{dest}/summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Saved {len(candidates)} candidates to Drive:")
print(f"  {dest}/")
print(f"  - candidate_pool.json")
print(f"  - summary.json")
print(f"\nNext: dock these through HADDOCK3 or AF2-Multimer")

In [ ]:
# ============================================================
# CELL 7: Compare to your manual candidates
# ============================================================
import json

with open('/content/current_run.txt') as f:
    RUN_DIR = f.read().strip()

candidates = json.load(open(f"{RUN_DIR}/candidates/candidate_pool.json"))

# Your manually designed candidates for comparison
manual = [
    {'id': 'MANUAL-1', 'sequence': 'DPETGEDF', 'source': 'manual'},
    {'id': 'MANUAL-2', 'sequence': 'DPETGEFF', 'source': 'manual'},
    {'id': 'MANUAL-3', 'sequence': 'DPETGEWY', 'source': 'manual'},
]

print(f"{'='*60}")
print(f"  ProteinMPNN 8-mers vs Your Manual Designs")
print(f"{'='*60}")
print(f"")
print(f"  Your manual candidates:")
for m in manual:
    match = any(c['sequence'] == m['sequence'] for c in candidates)
    print(f"    {m['sequence']}  {'ALSO FOUND BY MPNN' if match else 'unique to manual design'}")

print(f"\n  Top 10 ProteinMPNN candidates (by MPNN score):")
scored = [c for c in candidates if c.get('mpnn_score') is not None]
scored.sort(key=lambda c: c['mpnn_score'])
for c in scored[:10]:
    in_manual = any(c['sequence'] == m['sequence'] for m in manual)
    flag = ' *** MATCHES MANUAL' if in_manual else ''
    print(f"    {c['sequence']}  MPNN={c['mpnn_score']:.4f}{flag}")

print(f"\n  Novel sequences not in your manual designs:")
manual_seqs = set(m['sequence'] for m in manual)
novel = [c for c in scored if c['sequence'] not in manual_seqs]
print(f"    {len(novel)} novel candidates to explore")